In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType
)

print("DMart Big Data Analytics")
print("Spark version:", spark.version)


DMart Big Data Analytics
Spark version: 4.2.0


In [0]:

# Path of the uploaded DMart dataset
file_path = "/Volumes/workspace/default/dmart_files/DMart_sample_data.csv"

# Read CSV into a Spark DataFrame
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

print("DMart dataset loaded successfully!")

# Display total rows and columns
print("Total Rows:", df_raw.count())
print("Total Columns:", len(df_raw.columns))

# Display first 10 records
display(df_raw.limit(10))


DMart dataset loaded successfully!
Total Rows: 1000000
Total Columns: 8


Date,Time,CustomerType,ProductCategory,UnitPrice,Quantity,Total,FullName
31-12-2023 15:41,06:04:28,Member,Grocery,10.69,4,42.76,Jai Bhatia
07-01-2024 22:19,01:05:18,Premium,Home & Kitchen,381.03,3,1143.09,Ishaan Patel
10-01-2024 02:09,17:31:43,Regular,Grocery,153.84,5,769.2,Tanvi Iyer
01-06-2023 10:46,01:50:16,Regular,Electronics,341.41,4,1365.64,Zara Xavier
23-05-2024 22:07,10:37:06,Regular,Home & Kitchen,390.16,5,1950.8,Krishna Zutshi
16-02-2024 19:41,22:41:47,Member,Grocery,170.21,4,680.84,Zara Singh
30-11-2023 03:36,09:36:45,Member,Clothing,81.74,2,163.48,Eeshaan Xavier
05-06-2024 05:21,20:02:39,Premium,Clothing,181.26,3,543.78,Eeshaan Kumar
07-06-2024 21:40,21:07:44,Premium,Electronics,471.89,2,943.78,Mahesh Mehta
29-12-2023 04:35,17:53:54,Member,Grocery,59.7,2,119.4,Ishaan Verma


In [0]:

# Check the dataset schema
df_raw.printSchema()


root
 |-- Date: string (nullable = true)
 |-- Time: time(6) (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Total: double (nullable = true)
 |-- FullName: string (nullable = true)



In [0]:

from pyspark.sql import functions as F

# Check missing values in all columns
missing_values = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])

display(missing_values)


Date,Time,CustomerType,ProductCategory,UnitPrice,Quantity,Total,FullName
0,0,0,0,0,0,0,0


In [0]:

# Check duplicate records
total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)


Total Rows: 1000000
Unique Rows: 1000000
Duplicate Rows: 0


In [0]:

# Check invalid sales values
invalid_sales = df_raw.filter(
    (F.col("UnitPrice") <= 0) |
    (F.col("Quantity") <= 0) |
    (F.col("Total") < 0)
)

print("Total Rows:", df_raw.count())
print("Invalid Sales Records:", invalid_sales.count())


Total Rows: 1000000
Invalid Sales Records: 0


In [0]:

from pyspark.sql import functions as F

# Convert Date string into timestamp
df_clean = df_raw.withColumn(
    "TransactionTimestamp",
    F.to_timestamp("Date", "dd-MM-yyyy HH:mm")
)

# Display original and converted dates
display(
    df_clean.select(
        "Date",
        "TransactionTimestamp"
    ).limit(10)
)


Date,TransactionTimestamp
31-12-2023 15:41,2023-12-31T15:41:00.000Z
07-01-2024 22:19,2024-01-07T22:19:00.000Z
10-01-2024 02:09,2024-01-10T02:09:00.000Z
01-06-2023 10:46,2023-06-01T10:46:00.000Z
23-05-2024 22:07,2024-05-23T22:07:00.000Z
16-02-2024 19:41,2024-02-16T19:41:00.000Z
30-11-2023 03:36,2023-11-30T03:36:00.000Z
05-06-2024 05:21,2024-06-05T05:21:00.000Z
07-06-2024 21:40,2024-06-07T21:40:00.000Z
29-12-2023 04:35,2023-12-29T04:35:00.000Z


In [0]:

# Check if any dates failed conversion
invalid_dates = df_clean.filter(
    F.col("TransactionTimestamp").isNull()
).count()

print("Total Rows:", df_clean.count())
print("Invalid Dates:", invalid_dates)


Total Rows: 1000000
Invalid Dates: 0


In [0]:

# Create new date-related features
df_clean = (
    df_clean
    .withColumn(
        "TransactionDate",
        F.to_date("TransactionTimestamp")
    )
    .withColumn(
        "Year",
        F.year("TransactionTimestamp")
    )
    .withColumn(
        "Month",
        F.month("TransactionTimestamp")
    )
    .withColumn(
        "DayOfWeek",
        F.dayofweek("TransactionTimestamp")
    )
    .withColumn(
        "Hour",
        F.hour("TransactionTimestamp")
    )
)

# Display the transformed dataset
display(
    df_clean.select(
        "Date",
        "TransactionTimestamp",
        "TransactionDate",
        "Year",
        "Month",
        "DayOfWeek",
        "Hour"
    ).limit(10)
)


Date,TransactionTimestamp,TransactionDate,Year,Month,DayOfWeek,Hour
31-12-2023 15:41,2023-12-31T15:41:00.000Z,2023-12-31,2023,12,1,15
07-01-2024 22:19,2024-01-07T22:19:00.000Z,2024-01-07,2024,1,1,22
10-01-2024 02:09,2024-01-10T02:09:00.000Z,2024-01-10,2024,1,4,2
01-06-2023 10:46,2023-06-01T10:46:00.000Z,2023-06-01,2023,6,5,10
23-05-2024 22:07,2024-05-23T22:07:00.000Z,2024-05-23,2024,5,5,22
16-02-2024 19:41,2024-02-16T19:41:00.000Z,2024-02-16,2024,2,6,19
30-11-2023 03:36,2023-11-30T03:36:00.000Z,2023-11-30,2023,11,5,3
05-06-2024 05:21,2024-06-05T05:21:00.000Z,2024-06-05,2024,6,4,5
07-06-2024 21:40,2024-06-07T21:40:00.000Z,2024-06-07,2024,6,6,21
29-12-2023 04:35,2023-12-29T04:35:00.000Z,2023-12-29,2023,12,6,4


In [0]:

print("Original Rows:", df_raw.count())
print("Transformed Rows:", df_clean.count())
print("Total Columns:", len(df_clean.columns))

df_clean.printSchema()


Original Rows: 1000000
Transformed Rows: 1000000
Total Columns: 14
root
 |-- Date: string (nullable = true)
 |-- Time: time(6) (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Total: double (nullable = true)
 |-- FullName: string (nullable = true)
 |-- TransactionTimestamp: timestamp (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Hour: integer (nullable = true)



In [0]:

# Define the Delta table name
table_name = "workspace.default.dmart_sales_clean"

# Save transformed dataset in Delta format
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print("Delta Lake table created successfully!")
print("Table Name:", table_name)


Delta Lake table created successfully!
Table Name: workspace.default.dmart_sales_clean


In [0]:

# Read the saved Delta table
df_delta = spark.table(table_name)

print("Delta Table Rows:", df_delta.count())
print("Delta Table Columns:", len(df_delta.columns))

display(df_delta.limit(10))


Delta Table Rows: 1000000
Delta Table Columns: 14


Date,Time,CustomerType,ProductCategory,UnitPrice,Quantity,Total,FullName,TransactionTimestamp,TransactionDate,Year,Month,DayOfWeek,Hour
25-06-2023 04:57,08:54:48,Member,Clothing,43.88,1,43.88,Yash Iyer,2023-06-25T04:57:00.000Z,2023-06-25,2023,6,1,4
14-05-2023 06:26,03:32:29,Member,Electronics,125.33,1,125.33,Priya Gupta,2023-05-14T06:26:00.000Z,2023-05-14,2023,5,1,6
06-08-2023 21:49,20:24:44,Member,Home & Kitchen,188.99,5,944.95,Shreya Singh,2023-08-06T21:49:00.000Z,2023-08-06,2023,8,1,21
29-01-2023 15:03,13:04:34,Premium,Electronics,197.98,5,989.9,Aakash Desai,2023-01-29T15:03:00.000Z,2023-01-29,2023,1,1,15
07-10-2024 02:31,06:31:51,Member,Electronics,134.84,4,539.36,Tanvi Sharma,2024-10-07T02:31:00.000Z,2024-10-07,2024,10,2,2
05-05-2023 07:14,05:41:26,Premium,Electronics,458.39,2,916.78,Lakshmi Iyer,2023-05-05T07:14:00.000Z,2023-05-05,2023,5,6,7
31-12-2024 15:52,01:08:07,Premium,Clothing,414.43,1,414.43,Nandini Mehta,2024-12-31T15:52:00.000Z,2024-12-31,2024,12,3,15
30-12-2024 00:50,22:09:43,Premium,Home & Kitchen,197.18,1,197.18,Vaibhav Mehta,2024-12-30T00:50:00.000Z,2024-12-30,2024,12,2,0
15-09-2023 11:35,08:18:16,Regular,Electronics,37.23,1,37.23,Rohan Singh,2023-09-15T11:35:00.000Z,2023-09-15,2023,9,6,11
10-07-2023 06:15,01:03:12,Regular,Beauty & Personal Care,238.4,5,1192.0,Aakash Patel,2023-07-10T06:15:00.000Z,2023-07-10,2023,7,2,6


In [0]:

display(
    spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    )
)


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ad23ca8f-9045-4ca7-99ca-3f49cbf87d33,workspace.default.dmart_sales_clean,null,,2026-10-14T04:05:58.945Z,2026-10-26T04:46:21.000Z,List(),List(),2,27715446,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 90dded40-0386-4542-8ea6-c5d8d166d7c7)",3,7,"List(appendOnly, deletionVectors, invariants, timeType-preview)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:

# Read the Delta Lake table
df_delta = spark.table(
    "workspace.default.dmart_sales_clean"
)

# Verify saved data
print("Total Rows:", df_delta.count())
print("Total Columns:", len(df_delta.columns))

# Display sample records
display(df_delta.limit(10))


Total Rows: 1000000
Total Columns: 14


Date,Time,CustomerType,ProductCategory,UnitPrice,Quantity,Total,FullName,TransactionTimestamp,TransactionDate,Year,Month,DayOfWeek,Hour
25-06-2023 04:57,08:54:48,Member,Clothing,43.88,1,43.88,Yash Iyer,2023-06-25T04:57:00.000Z,2023-06-25,2023,6,1,4
14-05-2023 06:26,03:32:29,Member,Electronics,125.33,1,125.33,Priya Gupta,2023-05-14T06:26:00.000Z,2023-05-14,2023,5,1,6
06-08-2023 21:49,20:24:44,Member,Home & Kitchen,188.99,5,944.95,Shreya Singh,2023-08-06T21:49:00.000Z,2023-08-06,2023,8,1,21
29-01-2023 15:03,13:04:34,Premium,Electronics,197.98,5,989.9,Aakash Desai,2023-01-29T15:03:00.000Z,2023-01-29,2023,1,1,15
07-10-2024 02:31,06:31:51,Member,Electronics,134.84,4,539.36,Tanvi Sharma,2024-10-07T02:31:00.000Z,2024-10-07,2024,10,2,2
05-05-2023 07:14,05:41:26,Premium,Electronics,458.39,2,916.78,Lakshmi Iyer,2023-05-05T07:14:00.000Z,2023-05-05,2023,5,6,7
31-12-2024 15:52,01:08:07,Premium,Clothing,414.43,1,414.43,Nandini Mehta,2024-12-31T15:52:00.000Z,2024-12-31,2024,12,3,15
30-12-2024 00:50,22:09:43,Premium,Home & Kitchen,197.18,1,197.18,Vaibhav Mehta,2024-12-30T00:50:00.000Z,2024-12-30,2024,12,2,0
15-09-2023 11:35,08:18:16,Regular,Electronics,37.23,1,37.23,Rohan Singh,2023-09-15T11:35:00.000Z,2023-09-15,2023,9,6,11
10-07-2023 06:15,01:03:12,Regular,Beauty & Personal Care,238.4,5,1192.0,Aakash Patel,2023-07-10T06:15:00.000Z,2023-07-10,2023,7,2,6


In [0]:

# Task 1 - B5 - Cell 1
# Verify Delta Lake table and storage statistics

table_name = "workspace.default.dmart_sales_clean"

table_detail = spark.sql(f"DESCRIBE DETAIL {table_name}")

print("B5 - DELTA LAKE TABLE VERIFICATION")

display(
    table_detail.select(
        "format",
        "numFiles",
        "sizeInBytes"
    )
)

df_delta = spark.table(table_name)

print("Total Rows:", df_delta.count())
print("Total Columns:", len(df_delta.columns))


B5 - DELTA LAKE TABLE VERIFICATION


format,numFiles,sizeInBytes
delta,1,27457997


Total Rows: 1000000
Total Columns: 14


In [0]:

# Task 1 - B5 - Cell 2
# Execute Delta Lake OPTIMIZE

print("B5 - DELTA LAKE OPTIMISATION")

optimize_result = spark.sql(
    f"OPTIMIZE {table_name}"
)

# Show path and complete optimisation metrics
optimize_result.show(truncate=False)

# Interactive Databricks output
display(optimize_result)


B5 - DELTA LAKE OPTIMISATION
+----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path|metrics                                                                                                                                                                             |
+----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|    |{0, 0, {NULL, NULL, 0.0, 0, 0}, {NULL, NULL, 0.0, 0, 0}, 0, NULL, NULL, 0, 0, 1, 1, true, 0, 0, 1792992648927, 1792992650901, 8, 0, NULL, {0, 0}, NULL, 14, 14, 0, 0, NULL, NULL, 0}|
+----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1792992648927, 1792992650901, 8, 0, null, List(0, 0), null, 14, 14, 0, 0, null, null, 0)"


In [0]:

# Task 1 - B5 - Cell 3
# Inspect file compaction metrics

print("B5 - OPTIMISATION METRICS")

metrics_df = optimize_result.select("metrics.*")

display(metrics_df)

metrics = metrics_df.first().asDict(recursive=True)

files_added = metrics.get("numFilesAdded", 0)
files_removed = metrics.get("numFilesRemoved", 0)

print("Files Added:", files_added)
print("Files Removed:", files_removed)

if files_added == 0 and files_removed == 0:
    print(
        "No file rewriting was recorded. "
        "The table may not require further compaction."
    )
else:
    print(
        "File rewriting was recorded during OPTIMIZE."
    )


Project: DMart Big Data Analytics
Spark Version: 4.2.0
Storage Format: Delta Lake
Table: workspace.default.dmart_sales_clean
Shuffle Partitions: auto


In [0]:

# Task 1 - B5 - Cell 4
# Verify table after OPTIMIZE

print("B5 - POST-OPTIMISATION VERIFICATION")

after_detail = spark.sql(
    f"DESCRIBE DETAIL {table_name}"
)

display(
    after_detail.select(
        "format",
        "numFiles",
        "sizeInBytes"
    )
)

df_verified = spark.table(table_name)

print("Verified Rows:", df_verified.count())
print("Verified Columns:", len(df_verified.columns))


B5 - POST-OPTIMISATION VERIFICATION


format,numFiles,sizeInBytes
delta,1,27457997


Verified Rows: 1000000
Verified Columns: 14


In [0]:

# Task 1 - B5 - Cell 5
# Repeated query performance benchmark

import time
import statistics
from pyspark.sql import functions as F

def benchmark_query():
    df = spark.table(table_name)

    result = (
        df.groupBy("ProductCategory")
        .agg(
            F.round(F.sum("Total"), 2).alias("TotalSales"),
            F.sum("Quantity").alias("TotalQuantity"),
            F.count("*").alias("Transactions")
        )
    )

    return result.collect()

print("B5 - QUERY PERFORMANCE BENCHMARK")

# Warm-up execution
benchmark_query()

execution_times = []

for run in range(5):
    start = time.perf_counter()

    rows = benchmark_query()

    elapsed = time.perf_counter() - start
    execution_times.append(elapsed)

    print(f"Run {run + 1}: {elapsed:.4f} seconds")

median_time = statistics.median(execution_times)

print("\nPERFORMANCE SUMMARY")
print(f"Median Execution Time: {median_time:.4f} seconds")
print(f"Output Categories: {len(rows)}")


B5 - QUERY PERFORMANCE BENCHMARK
Run 1: 0.6620 seconds
Run 2: 0.6564 seconds
Run 3: 0.7195 seconds
Run 4: 0.7557 seconds
Run 5: 0.7045 seconds

PERFORMANCE SUMMARY
Median Execution Time: 0.7045 seconds
Output Categories: 5


In [0]:

# Task 1 - B5 - Cell 6
# Verify Apache Spark and Delta Lake configuration

print("B5 - SPARK ENVIRONMENT CONFIGURATION")

print("Project: DMart Big Data Analytics")
print("Spark Version:", spark.version)
print("Storage Format: Delta Lake")
print("Table:", table_name)

print(
    "Shuffle Partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

print(
    "Compute: Verify Serverless CPU in Databricks UI"
)


B5 - SPARK ENVIRONMENT CONFIGURATION
Project: DMart Big Data Analytics
Spark Version: 4.2.0
Storage Format: Delta Lake
Table: workspace.default.dmart_sales_clean
Shuffle Partitions: auto
Compute: Verify Serverless CPU in Databricks UI


In [0]:

# Read the cleaned Delta table
df_delta = spark.table(
    "workspace.default.dmart_sales_clean"
)

# Display Spark execution plan
print("DMart Delta Table Execution Plan")

df_delta.explain(mode="formatted")


DMart Delta Table Execution Plan
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet workspace.default.dmart_sales_clean (1)


(1) Scan parquet workspace.default.dmart_sales_clean
Output [15]: [Date#11581, Time#11582, CustomerType#11583, ProductCategory#11584, UnitPrice#11585, Quantity#11586, Total#11587, FullName#11588, TransactionTimestamp#11589, TransactionDate#11590, Year#11591, Month#11592, DayOfWeek#11593, Hour#11594, _databricks_internal_edge_computed_column_skip_row#11595]
Batched: true
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-zm7qas0dn2/uc/8bca585b-2624-4449-a326-c04ddcc0ba68/5f200a07-a276-43f4-bd45-b92f90078c9f/__unitystorage/catalogs/b9b1a287-547a-4f3e-9943-9156cbb0f8d7/tables/90dded40-0386-4542-8ea6-c5d8d166d7c7]
ReadSchema: struct<Date:string,Time:time(6),CustomerType:string,ProductCategory:string,UnitPrice:double,Quantity:int,Total:double,FullName:string,TransactionTimestamp:timestamp,TransactionDate:date,Year:int

In [0]:

# Read the cleaned Delta table
df_delta = spark.table(
    "workspace.default.dmart_sales_clean"
)

# Display Spark execution plan
print("DMart Delta Table Execution Plan")

df_delta.explain(mode="formatted")


DMart Delta Table Execution Plan
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet workspace.default.dmart_sales_clean (1)


(1) Scan parquet workspace.default.dmart_sales_clean
Output [15]: [Date#11640, Time#11641, CustomerType#11642, ProductCategory#11643, UnitPrice#11644, Quantity#11645, Total#11646, FullName#11647, TransactionTimestamp#11648, TransactionDate#11649, Year#11650, Month#11651, DayOfWeek#11652, Hour#11653, _databricks_internal_edge_computed_column_skip_row#11654]
Batched: true
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-zm7qas0dn2/uc/8bca585b-2624-4449-a326-c04ddcc0ba68/5f200a07-a276-43f4-bd45-b92f90078c9f/__unitystorage/catalogs/b9b1a287-547a-4f3e-9943-9156cbb0f8d7/tables/90dded40-0386-4542-8ea6-c5d8d166d7c7]
ReadSchema: struct<Date:string,Time:time(6),CustomerType:string,ProductCategory:string,UnitPrice:double,Quantity:int,Total:double,FullName:string,TransactionTimestamp:timestamp,TransactionDate:date,Year:int

In [0]:

from pyspark.sql import functions as F

# Repartition the dataset into 4 partitions
df_partitioned = df_delta.repartition(4)

# Calculate records in each partition
partition_counts = (
    df_partitioned
    .withColumn(
        "PartitionID",
        F.spark_partition_id()
    )
    .groupBy("PartitionID")
    .count()
    .orderBy("PartitionID")
)

display(partition_counts)


PartitionID,count
0,250000
1,250000
2,250000
3,250000
